# CE49X Lab 2: Is Wave Energy Worth the Investment?
## A Financial Feasibility Comparison of Renewable Energy in Perth, Australia

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

## Background

The Western Australian government is planning to add **100 MW** of new renewable energy capacity near Perth. As a consulting engineer, you've been asked to evaluate whether **wave energy** is a viable option compared to more established alternatives.

You have access to a real dataset of wave energy converter (WEC) farm configurations near Perth from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/882/large-scale+wave+energy+farm). The dataset (`WEC_Perth_49.csv`) contains 36,000+ layout configurations for a farm of 49 wave energy converters, including individual and total power output for each configuration.

Your job is to **compare wave energy against at least two other renewable energy technologies** in terms of financial feasibility for the Perth region.

## Deliverables

Your notebook must include the following:

### 1. Wave Energy Analysis (from the dataset)
- Load and explore the `WEC_Perth_49.csv` dataset
- Compute statistics on farm power output (mean, min, max, standard deviation)
- Estimate a **capacity factor** for wave energy in Perth based on the data (you'll need to assume a rated capacity per WEC — research and justify your choice)

### 2. Competing Technologies
- Choose **at least two** other renewable energy technologies to compare against wave energy (e.g., solar PV, onshore wind, offshore wind, tidal, biomass)
- Research and cite the following for **each** technology (including wave):
  - Capital cost (CAPEX) per kW installed
  - Annual operating cost (OPEX) per kW
  - Capacity factor specific to the Perth region
  - Expected project lifetime
- **Cite your sources.** Use data from reputable organizations (e.g., IRENA, IEA, CSIRO, NREL, Lazard).

### 3. Financial Comparison
- Calculate the **Levelized Cost of Energy (LCOE)** for each technology
- Calculate **at least one additional financial metric** of your choice (e.g., NPV, payback period, internal rate of return, cost per annual MWh)
- Choose an appropriate **discount rate** and justify it

### 4. Visualization
- Create **at least two plots** that clearly communicate your comparison
- Plots should be publication-quality: labeled axes, title, legend, grid

### 5. Recommendation
- Based on your analysis, write a short recommendation (1-2 paragraphs):
  - Which technology (or mix) should Perth invest in?
  - Under what conditions could wave energy become competitive?
  - What factors does your financial model **not** capture?

## Hints

- **LCOE formula:**

$$\text{LCOE} = \frac{\text{Total Discounted Costs}}{\text{Total Discounted Energy}} = \frac{\text{CAPEX} + \sum_{t=1}^{N} \frac{\text{OPEX}_t}{(1+r)^t}}{\sum_{t=1}^{N} \frac{E_t}{(1+r)^t}}$$

  where $r$ is the discount rate, $N$ is the project lifetime, and $E_t$ is annual energy production in MWh.

- **Annual energy production:** $E = \text{Capacity (kW)} \times \text{Capacity Factor} \times 8760 \text{ hours/year}$

- The dataset gives power in **Watts**. Be careful with unit conversions.

- Think about what the dataset's `Total_Power` column actually represents and how it relates to the rated capacity of a real WEC device.

## Grading

| Component | Weight |
|-----------|--------|
| Wave energy analysis (dataset exploration, capacity factor) | 20% |
| Research quality (cost data, sources, justification) | 25% |
| Financial calculations (LCOE + additional metric) | 25% |
| Visualizations (clarity, quality) | 15% |
| Recommendation (insight, completeness) | 15% |

## Submission

1. Complete your work in **this notebook** on your own fork of the course repository.
2. Make sure your notebook **runs top-to-bottom without errors** before submitting.
3. Commit and push your completed notebook to your fork.
4. We will grade directly from your fork — there is no separate upload. Make sure your latest work is pushed before the deadline.

---
## Your Work Starts Here

In [ ]:
# Comparative Techno-Economic Assessment of Wave Energy in Perth
# Paste this entire script into Cursor or a Jupyter cell/script file.
# Dataset path: WEC_Perth_49.csv is in Week02_Python_Modules_and_Data_Science/ (parent of lab/).

import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from textwrap import dedent

plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.grid"] = True

# ============================================================
# 1) INTRODUCTION
# ============================================================

intro = dedent("""
Comparative Techno-Economic Assessment of Wave Energy in Perth

This study evaluates the techno-economic performance of wave energy in Perth
using the WEC_Perth_49.csv dataset and compares it against two competing
renewable technologies: utility-scale solar PV and onshore wind.

The analysis includes:
1. Loading and exploring the Perth wave dataset,
2. Computing descriptive statistics for farm power output,
3. Estimating a wave-energy capacity factor,
4. Comparing CAPEX, OPEX, capacity factor, and lifetime across technologies,
5. Calculating LCOE and NPV,
6. Producing comparison plots,
7. Writing a final recommendation.

Source basis used in the assumptions:
- UCI Large-scale Wave Energy Farm dataset for Perth farm output.
- AEMO/Aurecon 2024 energy technology review for wave, solar PV, and onshore wind
  techno-economic assumptions.
- ARENA/UWA wave energy report for a Perth-relevant 1 MW/WEC assumption.
""")
print(intro)

# ============================================================
# 2) LOAD AND EXPLORE THE WAVE DATASET
# ============================================================

def _find_wec_csv(name: str = "WEC_Perth_49.csv") -> Path:
    """Resolve CSV path whether the kernel cwd is lab/, Week02..., or repo root."""
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        for candidate in (
            base / name,
            base / "Week02_Python_Modules_and_Data_Science" / name,
        ):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(
        f'{name} not found under {start}. '
        "Ensure the CE49X repo includes Week02_Python_Modules_and_Data_Science/WEC_Perth_49.csv "
        "or place the CSV next to this notebook."
    )

_csv_path = _find_wec_csv()
print(f"Loading dataset from: {_csv_path}")
df = pd.read_csv(_csv_path)

print("\n================ DATASET OVERVIEW ================\n")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

candidate_total_cols = [c for c in df.columns if "total" in c.lower() and "power" in c.lower()]
if len(candidate_total_cols) == 0:
    raise ValueError(
        "No total power column was detected automatically.\n"
        "Inspect df.columns and manually set total_col."
    )

total_col = candidate_total_cols[0]
farm_power = df[total_col].copy()

print("\nSelected total power column:", total_col)
print("\nDescriptive statistics for total farm power:")
print(farm_power.describe())

# ============================================================
# 3) WAVE FARM POWER STATISTICS
# ============================================================

wave_mean = farm_power.mean()
wave_min = farm_power.min()
wave_max = farm_power.max()
wave_std = farm_power.std()

wave_stats = pd.DataFrame({
    "Metric": ["Mean", "Minimum", "Maximum", "Standard Deviation"],
    "Value": [wave_mean, wave_min, wave_max, wave_std]
})

print("\n================ WAVE POWER STATISTICS ================\n")
print(wave_stats.to_string(index=False))

# ============================================================
# 4) CAPACITY FACTOR ESTIMATION FOR WAVE ENERGY
# ============================================================

# We infer the dataset unit from the magnitude of values.
# This is necessary because capacity factor requires consistent power units.

if wave_max > 1e6:
    guessed_unit = "W"
elif wave_max > 1e3:
    guessed_unit = "kW"
else:
    guessed_unit = "MW"

print("\nGuessed farm power unit based on magnitude:", guessed_unit)

if guessed_unit == "W":
    avg_power_mw = wave_mean / 1e6
elif guessed_unit == "kW":
    avg_power_mw = wave_mean / 1e3
else:
    avg_power_mw = wave_mean

n_wec = 49

# Main assumption:
# 1 MW per WEC, based on Perth-relevant CETO-style interpretation
rated_per_wec_main_mw = 1.0

# Sensitivity case:
# 0.2 MW per WEC, consistent with AEMO generic wave reference project
rated_per_wec_sens_mw = 0.2

farm_rated_main_mw = n_wec * rated_per_wec_main_mw
farm_rated_sens_mw = n_wec * rated_per_wec_sens_mw

cf_wave_main = avg_power_mw / farm_rated_main_mw
cf_wave_sens = avg_power_mw / farm_rated_sens_mw

cf_results = pd.DataFrame({
    "Scenario": ["Main case: 1 MW/WEC", "Sensitivity: 0.2 MW/WEC"],
    "Average Farm Output (MW)": [avg_power_mw, avg_power_mw],
    "Rated Farm Capacity (MW)": [farm_rated_main_mw, farm_rated_sens_mw],
    "Capacity Factor": [cf_wave_main, cf_wave_sens]
})

print("\n================ WAVE CAPACITY FACTOR ================\n")
print(cf_results.to_string(index=False))

# Use main dataset-based CF unless it is physically impossible (>1), in which case
# revert to literature benchmark of 35%.
cf_wave_used = cf_wave_main
if cf_wave_used > 1:
    print("\nWarning: dataset-based CF > 1.0, so literature benchmark CF = 0.35 will be used.")
    cf_wave_used = 0.35

print("Wave CF used in financial calculations:", cf_wave_used)

# ============================================================
# 5) COMPETING TECHNOLOGIES
# ============================================================

# Assumptions:
# Wave Energy:
#   CAPEX = 14,670 AUD/kW
#   OPEX = 520,000 AUD/MW-year = 520 AUD/kW-year
#   Lifetime = 25 years
#   CF = dataset-based or benchmark
#
# Solar PV:
#   CAPEX = 1,150 AUD/kW
#   OPEX = 12,000 AUD/MW-year = 12 AUD/kW-year
#   Lifetime = 30 years
#   CF = 29%
#
# Onshore Wind:
#   CAPEX = 3,050 AUD/kW
#   OPEX = 28,000 AUD/MW-year = 28 AUD/kW-year
#   Lifetime = 25 years
#   CF = 36%

tech_data = pd.DataFrame([
    {
        "Technology": "Wave Energy",
        "CAPEX_AUD_per_kW": 14670,
        "OPEX_AUD_per_kWyr": 520000 / 1000,
        "CapacityFactor": cf_wave_used,
        "Lifetime_yr": 25
    },
    {
        "Technology": "Solar PV",
        "CAPEX_AUD_per_kW": 1150,
        "OPEX_AUD_per_kWyr": 12000 / 1000,
        "CapacityFactor": 0.29,
        "Lifetime_yr": 30
    },
    {
        "Technology": "Onshore Wind",
        "CAPEX_AUD_per_kW": 3050,
        "OPEX_AUD_per_kWyr": 28000 / 1000,
        "CapacityFactor": 0.36,
        "Lifetime_yr": 25
    }
])

print("\n================ TECHNOLOGY INPUTS ================\n")
print(tech_data.to_string(index=False))

# ============================================================
# 6) FINANCIAL COMPARISON
# ============================================================

# Discount rate assumption
discount_rate = 0.07

def crf(r, n):
    return r * (1 + r)**n / ((1 + r)**n - 1)

def lcoe_aud_per_mwh(capex_kw, opex_kwyr, cf, n, r=0.07):
    annual_energy_mwh_per_kw = 8.76 * cf
    return (capex_kw * crf(r, n) + opex_kwyr) / annual_energy_mwh_per_kw

# Additional financial metric: NPV per kW installed
power_price = 100  # AUD/MWh

def npv_per_kw(capex_kw, opex_kwyr, cf, n, price, r=0.07):
    annual_energy = 8.76 * cf
    annual_revenue = annual_energy * price
    annual_cashflow = annual_revenue - opex_kwyr
    return -capex_kw + sum(annual_cashflow / ((1 + r) ** t) for t in range(1, n + 1))

tech_data["Annual_MWh_per_kW"] = 8.76 * tech_data["CapacityFactor"]
tech_data["CRF"] = tech_data["Lifetime_yr"].apply(lambda n: crf(discount_rate, n))
tech_data["LCOE_AUD_per_MWh"] = tech_data.apply(
    lambda row: lcoe_aud_per_mwh(
        row["CAPEX_AUD_per_kW"],
        row["OPEX_AUD_per_kWyr"],
        row["CapacityFactor"],
        row["Lifetime_yr"],
        discount_rate
    ),
    axis=1
)
tech_data["NPV_AUD_per_kW"] = tech_data.apply(
    lambda row: npv_per_kw(
        row["CAPEX_AUD_per_kW"],
        row["OPEX_AUD_per_kWyr"],
        row["CapacityFactor"],
        row["Lifetime_yr"],
        power_price,
        discount_rate
    ),
    axis=1
)

print("\n================ FINANCIAL RESULTS ================\n")
print(tech_data[[
    "Technology",
    "CAPEX_AUD_per_kW",
    "OPEX_AUD_per_kWyr",
    "CapacityFactor",
    "Lifetime_yr",
    "Annual_MWh_per_kW",
    "LCOE_AUD_per_MWh",
    "NPV_AUD_per_kW"
]].round(3).to_string(index=False))

# ============================================================
# 7) VISUALIZATION
# ============================================================

plt.figure(figsize=(8, 5))
plt.bar(tech_data["Technology"], tech_data["LCOE_AUD_per_MWh"])
plt.title("LCOE Comparison of Renewable Technologies")
plt.xlabel("Technology")
plt.ylabel("LCOE (AUD/MWh)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(tech_data["CAPEX_AUD_per_kW"], tech_data["CapacityFactor"] * 100, s=120)

for _, row in tech_data.iterrows():
    plt.annotate(
        row["Technology"],
        (row["CAPEX_AUD_per_kW"], row["CapacityFactor"] * 100),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title("CAPEX vs Capacity Factor")
plt.xlabel("CAPEX (AUD/kW)")
plt.ylabel("Capacity Factor (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(farm_power, bins=40)
plt.title("Distribution of Perth Wave Farm Power Output")
plt.xlabel("Farm Power Output")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# 8) AUTOMATIC TEXT OUTPUT FOR THE REPORT
# ============================================================

wave_paragraph = f"""
Wave Energy Analysis:
The WEC_Perth_49.csv dataset was loaded and explored successfully. The selected
total farm power column was '{total_col}'. The mean farm power output is {wave_mean:.3f},
the minimum is {wave_min:.3f}, the maximum is {wave_max:.3f}, and the standard
deviation is {wave_std:.3f}. Based on the magnitude of the dataset values, the
farm power unit was inferred as {guessed_unit}. After conversion, the average
farm output is {avg_power_mw:.3f} MW.

For capacity factor estimation, a main-case assumption of 1 MW rated capacity
per WEC was used, giving a total rated farm capacity of {farm_rated_main_mw:.1f} MW.
This produces a dataset-based wave capacity factor of {cf_wave_main:.4f}. A
sensitivity check with 0.2 MW per WEC gives a capacity factor of {cf_wave_sens:.4f}.
The capacity factor used in the financial model is {cf_wave_used:.4f}.
"""

comparison_paragraph = f"""
Competing Technologies:
Wave energy was compared with utility-scale solar PV and onshore wind. The wave
energy assumptions used in the financial model are a CAPEX of 14,670 AUD/kW,
an annual OPEX of 520 AUD/kW-year, a project lifetime of 25 years, and a
capacity factor of {cf_wave_used:.4f}. Solar PV was assumed to have a CAPEX of
1,150 AUD/kW, OPEX of 12 AUD/kW-year, a lifetime of 30 years, and a capacity
factor of 0.29. Onshore wind was assumed to have a CAPEX of 3,050 AUD/kW,
OPEX of 28 AUD/kW-year, a lifetime of 25 years, and a capacity factor of 0.36.
"""

recommendation = """
Recommendation:
Based on the comparative LCOE and NPV results, Perth should currently prioritize
utility-scale solar PV and onshore wind over wave energy for large-scale
commercial deployment. Solar PV has the lowest capital cost and a strong LCOE
outcome, while onshore wind offers a higher capacity factor and larger annual
energy yield per kW installed. Together, these two technologies provide a more
economically attractive renewable portfolio than wave energy under present-day
cost assumptions.

Wave energy still has long-term strategic value for Perth and Western Australia
because of the region’s strong marine energy resource and the possibility of
complementing solar and wind generation. However, wave energy is not yet
financially competitive in this simplified model because its CAPEX and OPEX are
substantially higher than those of mature renewable technologies. Wave energy
could become more competitive if device costs fall, reliability improves,
maintenance requirements decline, and large multi-device arrays demonstrate
stable long-term performance. The financial model used here does not capture
all relevant factors, including grid integration benefits, resource diversity,
technology learning, policy support, environmental externalities, and broader
industrial spillover effects.
"""

print("\n================ REPORT TEXT ================\n")
print(wave_paragraph)
print(comparison_paragraph)
print(recommendation)

# ============================================================
# 9) REFERENCES
# ============================================================

print("\n================ REFERENCES USED ================\n")
print("1. UCI Machine Learning Repository - Large-scale Wave Energy Farm.")
print("2. AEMO / Aurecon - 2024 Energy Technology Cost and Technical Parameter Review.")
print("3. ARENA / UWA - Wave Energy Cost Reduction Resource Assessment Report.")


---

### Questions?

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr